# Task 1B: Advanced RAG Pipeline

Improvements over Naive RAG:
1. **3 chunking strategies**: Fixed, Recursive (markdown-aware), Layout-Aware (tables intact)
2. **Hybrid Search**: Vector + BM25 via Reciprocal Rank Fusion (RRF)
3. **Cross-Encoder Reranking**: BAAI/bge-reranker-v2-m3
4. **Query Rewriting**: LLM-based query reformulation

In [1]:
import sys
sys.path.insert(0, "..")

import json
from src.parsing import parse_all_pdfs
from src.chunking import chunk_fixed, chunk_recursive, chunk_layout_aware
from src.pipeline import RAGPipeline
from src.retrieval import dense_retrieve, BM25Retriever, hybrid_retrieve, rerank
from src.config import DEFAULT_CONFIG

In [2]:
parsed_texts = parse_all_pdfs(use_cache=True)
# Use one document for chunking comparison
sample_text = list(parsed_texts.values())[0][:5000]
sample_source = list(parsed_texts.keys())[0]

## 1. Compare 3 Chunking Strategies

In [3]:
for strategy_name, chunk_fn in [("Fixed", chunk_fixed), ("Recursive", chunk_recursive), ("Layout-Aware", chunk_layout_aware)]:
    chunks = chunk_fn(sample_text, sample_source, chunk_size=512, chunk_overlap=100)
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name} — {len(chunks)} chunks")
    print(f"{'='*60}")
    for i, c in enumerate(chunks[:3]):
        print(f"\n--- Chunk {i+1} ({len(c.page_content)} chars) ---")
        print(c.page_content[:200], "...")


Strategy: Fixed — 5 chunks

--- Chunk 1 (526 chars) ---
Одобрен                                Утвержден
решением Правления    решением Совета директоров
акционерного общества     акционерного общества
«Национальная компания    «Национальная компания
«Қаза ...

--- Chunk 2 (1533 chars) ---
| Ключевые показатели                                             | 2024    | 2023    | Абсолютное значение | %     |
| --------------------------------------------------------------- | ------- | ---- ...

--- Chunk 3 (1533 chars) ---
| Отправленные пассажиры, тыс. пассажиров                         | 13 796  | 13 681  | 115                 | 0,8   |
| Грузовые перевозки                                              |         |      ...

Strategy: Recursive — 5 chunks

--- Chunk 1 (526 chars) ---
Одобрен                                Утвержден
решением Правления    решением Совета директоров
акционерного общества     акционерного общества
«Национальная компания    «Национальная компания
«Қаза ...

-

## 2. Compare Retrieval Methods
Dense vs BM25 vs Hybrid on the same query.

In [4]:
# Build pipeline with layout-aware chunking
advanced_config = {
    **DEFAULT_CONFIG,
    "chunking_strategy": "layout_aware",
    "collection_name": "advanced_rag",
    "alpha": 0.5,
    "use_reranking": False,
    "use_query_rewriting": False,
}

pipeline = RAGPipeline(advanced_config)
n_chunks = pipeline.ingest(parsed_texts)
print(f"Indexed {n_chunks} chunks")

Indexed 282 chunks


In [5]:
test_query = "Какой объем доходов от грузовых перевозок получила КТЖ в 2024 году?"

print("=== Dense Retrieval ===")
dense_docs = dense_retrieve(pipeline.vector_store, test_query, top_k=3)
for i, doc in enumerate(dense_docs):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

print("\n=== BM25 Retrieval ===")
bm25_docs = pipeline.bm25_retriever.retrieve(test_query, top_k=3)
for i, doc in enumerate(bm25_docs):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

print("\n=== Hybrid Retrieval (alpha=0.5) ===")
hybrid_docs = hybrid_retrieve(pipeline.vector_store, pipeline.bm25_retriever, test_query, top_k=3, alpha=0.5)
for i, doc in enumerate(hybrid_docs):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

=== Dense Retrieval ===

Doc 1: # Ключевые достижения АО «НК «ҚТЖ» в реализации стратегических целей в 2024 году...

Doc 2: # «НАЦИОНАЛЬНАЯ КОМПАНИЯ «ҚАЗАҚСТАН ТЕМIР ЖОЛЫ» ЗА 2024 ГОД





КОМПАНИЯ В ЦИФРАХ...

Doc 3: # ГОДОВОЙ ОТЧЕТ 2024...

=== BM25 Retrieval ===

Doc 1: # Потоки прибыли

- Доходы от пассажирских перевозок
- Доходы от грузовых перевозок
- Доходы от экспедирования грузов и оперирования грузовыми вагонам...

Doc 2: # Цель 2 «Развитие транзитных перевозок»

Стабильный объем торговли между Европой и Азией, а также растущая востребованность железнодорожных перевозок...

Doc 3: # 4.1. Анализ нефтегазовой отрасли, макро - микроэкономические изменения

В 2024 году объем добычи нефти и газового конденсата в Республике Казахстан ...

=== Hybrid Retrieval (alpha=0.5) ===

Doc 1: # ГОДОВОЙ ОТЧЕТ 2024...

Doc 2: # Ключевые достижения АО «НК «ҚТЖ» в реализации стратегических целей в 2024 году...

Doc 3: # Потоки прибыли

- Доходы от пассажирских перевозок
- Доходы от грузовых пере

## 3. Reranking Effect

In [6]:
# Get top-10 hybrid results, then rerank to top-5
hybrid_10 = hybrid_retrieve(pipeline.vector_store, pipeline.bm25_retriever, test_query, top_k=10, alpha=0.5)
reranked = rerank(test_query, hybrid_10, top_k=5)

print("Before reranking (top 5):")
for i, doc in enumerate(hybrid_10[:5]):
    print(f"  {i+1}. {doc.page_content[:100]}...")

print("\nAfter reranking (top 5):")
for i, doc in enumerate(reranked):
    print(f"  {i+1}. {doc.page_content[:100]}...")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Before reranking (top 5):
  1. # ГОДОВОЙ ОТЧЕТ 2024...
  2. # Ключевые достижения АО «НК «ҚТЖ» в реализации стратегических целей в 2024 году...
  3. # Потоки прибыли

- Доходы от пассажирских перевозок
- Доходы от грузовых перевозок
- Доходы от эксп...
  4. # «НАЦИОНАЛЬНАЯ КОМПАНИЯ «ҚАЗАҚСТАН ТЕМIР ЖОЛЫ» ЗА 2024 ГОД





КОМПАНИЯ В ЦИФРАХ...
  5. # Цель 2 «Развитие транзитных перевозок»

Стабильный объем торговли между Европой и Азией, а также р...

After reranking (top 5):
  1. | Стратегические цели                               | Достижения 2024 года                          ...
  2. # Ключевые достижения АО «НК «ҚТЖ» в реализации стратегических целей в 2024 году...
  3. # Потоки прибыли

- Доходы от пассажирских перевозок
- Доходы от грузовых перевозок
- Доходы от эксп...
  4. # 6.2. Кредитный риск

Группа подвержена кредитному риску, который сопряжён с возможным неисполнение...
  5. # 4.3. Информации о продукции и реализации добываемой нефти

Выгодное географическое расположение ак

## 4. Query Rewriting Effect

In [7]:
pipeline_rewrite = RAGPipeline({**advanced_config, "use_query_rewriting": True})
pipeline_rewrite.vector_store = pipeline.vector_store
pipeline_rewrite.bm25_retriever = pipeline.bm25_retriever

original_q = "Сколько КТЖ заработал на грузоперевозках?"
rewritten_q = pipeline_rewrite._rewrite_query(original_q)

print(f"Original:  {original_q}")
print(f"Rewritten: {rewritten_q}")

# Compare retrieval results
docs_original = dense_retrieve(pipeline.vector_store, original_q, top_k=3)
docs_rewritten = dense_retrieve(pipeline.vector_store, rewritten_q, top_k=3)

print("\nOriginal query results:")
for i, doc in enumerate(docs_original):
    print(f"  {i+1}. {doc.page_content[:100]}...")

print("\nRewritten query results:")
for i, doc in enumerate(docs_rewritten):
    print(f"  {i+1}. {doc.page_content[:100]}...")

Original:  Сколько КТЖ заработал на грузоперевозках?
Rewritten: Каков объем дохода КТЖ от грузоперевозок?

Original query results:
  1. # Потоки прибыли

- Доходы от пассажирских перевозок
- Доходы от грузовых перевозок
- Доходы от эксп...
  2. # «НАЦИОНАЛЬНАЯ КОМПАНИЯ «ҚАЗАҚСТАН ТЕМIР ЖОЛЫ» ЗА 2024 ГОД





КОМПАНИЯ В ЦИФРАХ...
  3. # Объемы продаж и цены реализации нефти....

Rewritten query results:
  1. # Потоки прибыли

- Доходы от пассажирских перевозок
- Доходы от грузовых перевозок
- Доходы от эксп...
  2. # Обеспечивает качественную основу устойчивого роста бизнес-среды

Обеспечивает 64% грузооборота и 1...
  3. # «НАЦИОНАЛЬНАЯ КОМПАНИЯ «ҚАЗАҚСТАН ТЕМIР ЖОЛЫ» ЗА 2024 ГОД





КОМПАНИЯ В ЦИФРАХ...


## 5. Full Advanced Pipeline: Side-by-Side Comparison with Naive

In [8]:
# Naive pipeline
naive_config = {
    **DEFAULT_CONFIG,
    "chunking_strategy": "fixed",
    "alpha": 1.0,
    "use_reranking": False,
    "use_query_rewriting": False,
    "collection_name": "naive_rag",
}
naive_pipeline = RAGPipeline(naive_config)
naive_pipeline.ingest(parsed_texts)

# Advanced pipeline: hybrid + reranking + query rewriting
adv_full_config = {
    **DEFAULT_CONFIG,
    "chunking_strategy": "layout_aware",
    "alpha": 0.5,
    "use_reranking": True,
    "use_query_rewriting": True,
    "collection_name": "advanced_rag",
}
adv_pipeline = RAGPipeline(adv_full_config)
adv_pipeline.vector_store = pipeline.vector_store
adv_pipeline.bm25_retriever = pipeline.bm25_retriever
adv_pipeline.documents = pipeline.documents

In [9]:
with open("../data/golden_dataset.json", "r", encoding="utf-8") as f:
    golden = json.load(f)

# Compare on 10 questions
for item in golden[:10]:
    naive_result = naive_pipeline.query(item["question"])
    adv_result = adv_pipeline.query(item["question"])
    
    print(f"Q: {item['question']}")
    print(f"Expected: {item['ground_truth']}")
    print(f"Naive:    {naive_result['answer']}")
    print(f"Advanced: {adv_result['answer']}")
    print("=" * 80)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Q: Каков был доход от основной деятельности АО «НК «КТЖ» в 2024 году?
Expected: Доход от основной деятельности составил 2 163,9 млрд тенге.
Naive:    Консолидированная выручка АО «НК «ҚТЖ» в 2024 году составила 2 163,9 млрд тенге.
Advanced: В предоставленном контексте отсутствует информация о доходе от основной деятельности АО «НК «ҚТЖ» в 2024 году.


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Q: На сколько вырос доход от основной деятельности КТЖ в 2024 году по сравнению с 2023 годом (в абсолютном значении)?
Expected: Доход вырос на 229,8 млрд тенге.
Naive:    Консолидированная выручка АО «НК «ҚТЖ» в 2024 году составила 2 163,9 млрд тенге, что на 11,9% больше по сравнению с 2023 годом, когда выручка составила 1 937,5 млрд тенге (2 163,9 / 1,119 = 1 937,5). 

Таким образом, доход от основной деятельности КТЖ в 2024 году вырос на 226,4 млрд тенге (2 163,9 - 1 937,5).
Advanced: Доходы Компании от основной деятельности в 2024 году составили 2 163,9 млрд тенге, что на 11,9% выше уровня 2023 года. В абсолютном значении это составляет увеличение на 230,5 млрд тенге (2 163,9 млрд тенге - 1 933,4 млрд тенге, где 1 933,4 млрд тенге — это доходы за 2023 год, рассчитанные по формуле: 2 163,9 млрд тенге / 1,119).


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


...

Q: Какой процент роста грузооборота запланирован на 2025 год по сравнению с 2024?
Expected: Запланирован рост на 4,6%.
Naive:    Запланированный рост грузооборота на 2025 год по сравнению с 2024 годом составляет 4,6%.
Advanced: Запланированный процент роста грузооборота на 2025 год по сравнению с 2024 годом составляет 4,6%.


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Q: Какой целевой показатель по объему контейнерного транзита установлен на 2025 год?
Expected: Целевой показатель составляет 1 553 тыс. ДФЭ.
Naive:    Целевой показатель по объему контейнерного транзита на 2025 год установлен на уровне 1 553 тыс. ДФЭ, что составляет 11,3% к 2024 году.
Advanced: Целевой показатель по объему контейнерного транзита на 2025 год установлен на уровне 1 553 тыс. ДФЭ, что составляет 11,3% к 2024 году.


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Q: Какие крупные инфраструктурные объекты планирует ввести в эксплуатацию КТЖ в 2025 году?
Expected: Планируется ввод вторых путей на участке «Достык — Мойынты» и железнодорожной линии в обход станции Алматы.
Naive:    В 2025 году АО «НК «ҚТЖ» планирует ввести в эксплуатацию следующие крупные инфраструктурные объекты:

1. Второй ж/д путь на участке «Достык – Мойынты».
2. Железнодорожная линия в обход ст. Алматы.
Advanced: В 2025 году АО «НК «ҚТЖ» планирует ввести в эксплуатацию следующие крупные инфраструктурные объекты:

1. Второй ж/д путь на участке «Достык – Мойынты».
2. Железнодорожная линия в обход ст. Алматы.
